**INGEST **DATA****

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
APP_ID  = "9c21fe4c"     
APP_KEY = "162350ac09ba021df27b2a1e37cbad2e"    
COUNTRY = "us"               
CATEGORY = "it-jobs"
RESULTS_PER_PAGE = 50

In [0]:
import requests

url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/1"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": RESULTS_PER_PAGE,
    "category": CATEGORY,
    "content-type": "application/json",
}

resp = requests.get(url, params=params, timeout=30)

print("called URL:", resp.url)
print("response body:", resp.text[:500])

In [0]:
import requests

url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/1"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": RESULTS_PER_PAGE,
    "category": CATEGORY,
    "content-type": "application/json",
}

resp = requests.get(url, params=params, timeout=30)
print("status:", resp.status_code)

data = resp.json()
print("total jobs available:", data.get("count"))
print("results on this page:", len(data.get("results", [])))

data["results"][0]

In [0]:
import json
from pyspark.sql import functions as F

rows = [(json.dumps(r),) for r in data["results"]]
df = spark.createDataFrame(rows, "raw_json string")

df = (df
    .withColumn("ingestion_date", F.current_date())
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source", F.lit("adzuna")))

df.show(5, truncate=80)
print("rows:", df.count())

In [0]:
from datetime import date
today = date.today().isoformat()

(df.write
   .format("delta")
   .mode("overwrite")
   .option("replaceWhere", f"ingestion_date = '{today}'")
   .partitionBy("ingestion_date")
   .saveAsTable("workspace.bronze.adzuna_jobs_raw"))

print("written for", today)

In [0]:
spark.table("workspace.bronze.adzuna_jobs_raw").count()


In [0]:
display(spark.table("workspace.bronze.adzuna_jobs_raw"))